In [4]:
import os
import sys
import numpy as np

sys.path.append(
    os.path.abspath("CS195S2026/cs195-ska-projects/shared/ska_agent-1.0.0-8")
)

from ska_agent.core.pricing import PricingEngine
from ska_agent.core.structures import Segment
from ska_agent.models.embedding import Embedder
from ska_agent.evaluation.officeqa import OfficeQAQuestion


# ---- Constants ----
N_PER_MODE = 25


# ---- Helpers ----
def ensure_1d(array):
    arr = np.asarray(array, dtype=np.float64)
    return arr.reshape(-1)


def dollars(value):
    return f"${value:,.0f}"


def percent(value):
    return f"{value:.1f}%"


# ---- Paste your make_officeqa_synthetic_corpus function here ----
# (the one you already have, unchanged)
def make_officeqa_synthetic_corpus(n_per_mode=25):
    """
    Generate synthetic document segments and questions in OfficeQA style.

    Creates four question types:
      - LOOKUP:     retrieve a single value from one document
      - MULTI_DOC:  compare values across two documents
      - COMPUTE:    calculate a difference between two rows
      - MULTI_STEP: sum multiple values from one document

    Also creates distractor segments that look plausible but are wrong.

    Returns:
        segment_texts: list of raw text strings for each document chunk
        questions:     list of OfficeQAQuestion objects
    """
    rng = np.random.default_rng(7)

    agencies = [
        "Treasury Operations Office",
        "Public Debt Service",
        "Revenue Analysis Bureau",
        "Federal Grants Division",
        "Cash Management Office",
        "Budget Review Unit",
        "Infrastructure Finance Office",
        "Economic Stabilization Fund",
        "Intergovernmental Transfers Office",
        "Audit and Compliance Division",
    ]

    programs = [
        "Debt Servicing",
        "Emergency Grants",
        "Infrastructure Loans",
        "Tax Refund Processing",
        "Agency Payroll",
        "Treasury Securities",
        "Municipal Support",
        "Disaster Relief",
        "Technology Modernization",
        "Public Health Transfers",
    ]

    years = list(range(2015, 2025))

    segment_texts = []
    questions = []

    # --- LOOKUP questions ---
    # Each question asks for a single outlay value from one agency/program/year.
    for i in range(n_per_mode):
        agency = agencies[i % len(agencies)]
        program = programs[(i * 2) % len(programs)]
        year = years[i % len(years)]

        outlay = 5000 + 137 * i
        answer = dollars(outlay)

        segment_texts.append(
            f"[NODE: lookup_{i}_record] type=row | Year: {year} | Agency: {agency} | "
            f"Program: {program} | Outlays: {answer} | "
            f"The reported outlays for {program} under {agency} in {year} were {answer}."
        )

        # Add a nearby distractor with a slightly different value and year
        distractor_value = outlay + 913
        segment_texts.append(
            f"[NODE: lookup_{i}_distractor] type=row | Year: {year - 1} | Agency: {agency} | "
            f"Program: {program} Reserve | Outlays: {dollars(distractor_value)} | "
            f"This is a reserve entry and should not be used for the main {year} question."
        )

        questions.append(
            OfficeQAQuestion(
                question_id=f"lookup_{i}",
                question=f"What were the reported outlays for {program} under {agency} in {year}?",
                answer=answer,
                source_documents=[f"{agency}_{year}.pdf"],
                question_type="LOOKUP",
                difficulty="easy",
            )
        )

    # --- MULTI_DOC questions ---
    # Each question asks which of two agencies had larger obligations.
    for i in range(n_per_mode):
        agency_a = agencies[i % len(agencies)]
        agency_b = agencies[(i + 3) % len(agencies)]
        program = programs[(i + 1) % len(programs)]
        year = years[i % len(years)]

        value_a = 7000 + 91 * i
        value_b = 6500 + 83 * i

        if value_a > value_b:
            larger_agency = agency_a
        else:
            larger_agency = agency_b

        answer = larger_agency

        segment_texts.append(
            f"[NODE: multidoc_{i}_doc_a] type=row | Source: Bulletin A | Year: {year} | "
            f"Agency: {agency_a} | Program: {program} | Obligations: {dollars(value_a)}."
        )
        segment_texts.append(
            f"[NODE: multidoc_{i}_doc_b] type=row | Source: Bulletin B | Year: {year} | "
            f"Agency: {agency_b} | Program: {program} | Obligations: {dollars(value_b)}."
        )
        segment_texts.append(
            f"[NODE: multidoc_{i}_comparison] type=derived | Year: {year} | Program: {program} | "
            f"Comparing {agency_a} obligations of {dollars(value_a)} with "
            f"{agency_b} obligations of {dollars(value_b)}, "
            f"the agency with larger obligations was {answer}."
        )

        questions.append(
            OfficeQAQuestion(
                question_id=f"multidoc_{i}",
                question=(
                    f"Comparing {agency_a} and {agency_b}, which agency had larger obligations "
                    f"for {program} in {year}?"
                ),
                answer=answer,
                source_documents=[f"{agency_a}_{year}.pdf", f"{agency_b}_{year}.pdf"],
                question_type="MULTI_DOC",
                difficulty="medium",
            )
        )

    # --- COMPUTE questions ---
    # Each question asks for the year-over-year increase in receipts.
    for i in range(n_per_mode):
        agency = agencies[(i + 2) % len(agencies)]
        program = programs[(i + 4) % len(programs)]

        year_a = years[i % len(years)]
        year_b = year_a + 1 if year_a < 2024 else 2024

        base_receipts = 10000 + 111 * i
        new_receipts = base_receipts + 250 + 13 * (i % 9)
        difference = new_receipts - base_receipts
        answer = dollars(difference)

        segment_texts.append(
            f"[NODE: compute_{i}_year_a] type=row | Year: {year_a} | Agency: {agency} | "
            f"Program: {program} | Receipts: {dollars(base_receipts)}."
        )
        segment_texts.append(
            f"[NODE: compute_{i}_year_b] type=row | Year: {year_b} | Agency: {agency} | "
            f"Program: {program} | Receipts: {dollars(new_receipts)}."
        )
        segment_texts.append(
            f"[NODE: compute_{i}_derived] type=calculation | Agency: {agency} | "
            f"Program: {program} | "
            f"The increase in receipts from {year_a} to {year_b} was {answer}."
        )

        questions.append(
            OfficeQAQuestion(
                question_id=f"compute_{i}",
                question=(
                    f"What was the increase in receipts for {program} under {agency} "
                    f"from {year_a} to {year_b}?"
                ),
                answer=answer,
                source_documents=[f"{agency}_{year_a}.pdf", f"{agency}_{year_b}.pdf"],
                question_type="COMPUTE",
                difficulty="medium",
            )
        )

    # --- MULTI_STEP questions ---
    # Each question asks for the combined outlays of two programs under one agency.
    for i in range(n_per_mode):
        agency = agencies[(i + 5) % len(agencies)]
        program_a = programs[i % len(programs)]
        program_b = programs[(i + 5) % len(programs)]
        year = years[i % len(years)]

        amount_a = 8000 + 101 * i
        amount_b = 8500 + 77 * i
        total = amount_a + amount_b
        answer = dollars(total)

        segment_texts.append(
            f"[NODE: multistep_{i}_classification] type=metadata | Year: {year} | "
            f"Agency: {agency} | Priority programs: {program_a}; {program_b}."
        )
        segment_texts.append(
            f"[NODE: multistep_{i}_program_a] type=row | Year: {year} | Agency: {agency} | "
            f"Program: {program_a} | Priority Outlays: {dollars(amount_a)}."
        )
        segment_texts.append(
            f"[NODE: multistep_{i}_program_b] type=row | Year: {year} | Agency: {agency} | "
            f"Program: {program_b} | Priority Outlays: {dollars(amount_b)}."
        )
        segment_texts.append(
            f"[NODE: multistep_{i}_derived] type=calculation | Year: {year} | "
            f"Agency: {agency} | "
            f"The combined priority outlays for {program_a} and {program_b} were {answer}."
        )

        questions.append(
            OfficeQAQuestion(
                question_id=f"multistep_{i}",
                question=(
                    f"For {agency} in {year}, what were the combined priority outlays "
                    f"for {program_a} and {program_b}?"
                ),
                answer=answer,
                source_documents=[f"{agency}_{year}_priority.pdf"],
                question_type="MULTI_STEP",
                difficulty="hard",
            )
        )

    # --- Generic distractor segments ---
    # These are background notes not tied to any question.
    for i in range(50):
        agency = agencies[i % len(agencies)]
        program = programs[(i + 6) % len(programs)]
        year = years[i % len(years)]
        value = 3000 + 211 * i

        segment_texts.append(
            f"[NODE: distractor_{i}] type=note | Year: {year} | Agency: {agency} | "
            f"Program: {program} | Administrative note value: {dollars(value)} | "
            f"This note is background context and is not the final answer for the training questions."
        )

    return segment_texts, questions


# ---- Build corpus ONCE ----
print("Building corpus...")
embedder = Embedder(model_name="all-MiniLM-L6-v2")
segment_texts, questions = make_officeqa_synthetic_corpus(n_per_mode=N_PER_MODE)

print(f"Created {len(segment_texts)} raw segment texts.")
print(f"Created {len(questions)} OfficeQA-style questions.")

mode_counts = {}
for q in questions:
    mode_counts[q.question_type] = mode_counts.get(q.question_type, 0) + 1
print("Question counts by mode:")
for mode, count in mode_counts.items():
    print(f"  {mode}: {count}")

print("\nEmbedding segments...")
segment_vectors = embedder.embed(segment_texts)

segments = []
for i, (text, vec) in enumerate(zip(segment_texts, segment_vectors)):
    segments.append(Segment(
        text=text,
        vector=np.asarray(vec, dtype=np.float64),
        start_idx=i,
        end_idx=i + 1,
        sentences=[text],
        internal_cost=0.0,
    ))

print(f"Built {len(segments)} segments and {len(questions)} questions.\n")


# ============================================================
# Diagnostics
# ============================================================

# ---- Check 1: Embedding dimensions ----
print("=" * 60)
print("CHECK 1: Embedding dimension consistency")
print("=" * 60)

segment_vec_dim = segments[0].vector.shape
query_vec = embedder.embed_single("What were the reported outlays?")
query_vec_dim = np.asarray(query_vec).shape

print(f"  Segment.vector shape:         {segment_vec_dim}")
print(f"  embedder.embed_single shape:  {query_vec_dim}")

if segment_vec_dim == query_vec_dim:
    print("  ✓ Dimensions match")
else:
    print("  ✗ MISMATCH — this is a serious bug")


# ---- Check 2: Engine reuse safety ----
print("\n" + "=" * 60)
print("CHECK 2: Is reusing the engine safe?")
print("=" * 60)

test_query = questions[0].question

fresh_engine = PricingEngine(
    segments=segments, embed_fn=embedder.embed_single,
    lambda_sparsity=0.01, eta_redundancy=0.0, max_segments=5,
)
fresh_result = fresh_engine.retrieve(test_query, verbose=False)
fresh_ids = [s.start_idx for s in fresh_result.segments]

reused_engine = PricingEngine(
    segments=segments, embed_fn=embedder.embed_single,
    lambda_sparsity=0.5, eta_redundancy=0.0, max_segments=5,
)
_ = reused_engine.retrieve(questions[5].question, verbose=False)
reused_engine.lambda_sparsity = 0.01
reused_result = reused_engine.retrieve(test_query, verbose=False)
reused_ids = [s.start_idx for s in reused_result.segments]

print(f"  Fresh engine retrieved:  {fresh_ids}")
print(f"  Reused engine retrieved: {reused_ids}")
if fresh_ids == reused_ids:
    print("  ✓ Engine reuse is safe")
else:
    print("  ✗ Engine reuse changes results — revert to fresh-engine pattern")


# ---- Check 3: Gold segment ranking ----
print("\n" + "=" * 60)
print("CHECK 3: Gold segment ranking on a failing query")
print("=" * 60)

failing_q = questions[1]  # lookup_1
gold_marker = "lookup_1_record"

deep_engine = PricingEngine(
    segments=segments, embed_fn=embedder.embed_single,
    lambda_sparsity=0.001,
    eta_redundancy=0.0,
    max_segments=30,
)
deep_result = deep_engine.retrieve(failing_q.question, verbose=False)

print(f"  Query: {failing_q.question}")
print(f"  Gold answer: {failing_q.answer}")
print(f"  Looking for marker: {gold_marker}\n")

gold_rank = None
for rank, seg in enumerate(deep_result.segments):
    is_gold = gold_marker in seg.text
    marker = " <-- GOLD" if is_gold else ""
    print(f"    Rank {rank:2d}: {seg.text[:90]}{marker}")
    if is_gold and gold_rank is None:
        gold_rank = rank

print()
if gold_rank is None:
    print("  ✗ Gold segment NOT in top 30 — embedder is the problem")
elif gold_rank < 5:
    print(f"  ? Gold at rank {gold_rank} (within top 5) — bug is elsewhere")
elif gold_rank < 20:
    print(f"  ✗ Gold at rank {gold_rank} — max_segments=5 is too small")
else:
    print(f"  ✗ Gold at rank {gold_rank} — embedder weakly distinguishes it")


# ---- Check 4: Distribution of successes ----
print("\n" + "=" * 60)
print("CHECK 4: Distribution of successes across LOOKUP queries")
print("=" * 60)

quick_engine = PricingEngine(
    segments=segments, embed_fn=embedder.embed_single,
    lambda_sparsity=0.001, eta_redundancy=0.0, max_segments=20,
)

successes = []
failures = []
for q in questions:
    if q.question_type != "LOOKUP":
        continue
    r = quick_engine.retrieve(q.question, verbose=False)
    context = " ".join(s.text for s in r.segments).lower()
    found = q.answer.lower() in context
    (successes if found else failures).append(q.question_id)

print(f"  At lambda=0.001, max_segments=20:")
print(f"  Successes ({len(successes)}/25): {successes}")
print(f"  Failures  ({len(failures)}/25): {failures}")

Building corpus...
No GPU found. Using lightweight CPU embedder: Alibaba-NLP/gte-Qwen2-1.5B-instruct
Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/Users/kevincapcha/PycharmProjects/CS195S2026/cs195-ska-projects/shared/ska_agent-1.0.0-8/ska_agent/models/embedding.py:51: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()


Embedder ready (dim=384)
Created 350 raw segment texts.
Created 100 OfficeQA-style questions.
Question counts by mode:
  LOOKUP: 25
  MULTI_DOC: 25
  COMPUTE: 25
  MULTI_STEP: 25

Embedding segments...
Built 350 segments and 100 questions.

CHECK 1: Embedding dimension consistency
  Segment.vector shape:         (384,)
  embedder.embed_single shape:  (384,)
  ✓ Dimensions match

CHECK 2: Is reusing the engine safe?
  Fresh engine retrieved:  [0, 126, 260, 68]
  Reused engine retrieved: [0, 126, 260, 68]
  ✓ Engine reuse is safe

CHECK 3: Gold segment ranking on a failing query
  Query: What were the reported outlays for Infrastructure Loans under Public Debt Service in 2016?
  Gold answer: $5,137
  Looking for marker: lookup_1_record

    Rank  0: [NODE: lookup_1_record] type=row | Year: 2016 | Agency: Public Debt Service | Program: Inf <-- GOLD
    Rank  1: [NODE: compute_10_year_a] type=row | Year: 2015 | Agency: Revenue Analysis Bureau | Progra
    Rank  2: [NODE: distractor_6] type